# 1.0 - LOAD PACKAGES

## LOAD PACKAGES AND GET DATA

In [1]:
# 0.1 - DATA PREP AND MATH LIBRARIES
import pandas       as pd
import numpy        as np
import re

# PLOTTING
import seaborn      as sns
from   matplotlib                 import pyplot as plt
#%matplotlib inline 


# DECISION TREES
from   sklearn.feature_extraction import DictVectorizer        # CONVERT DATAFRAME TO MATRIX AND APPLY ONE-HOT ENCODING
from   sklearn.model_selection    import train_test_split      # SPLIT THE DATASET
from   sklearn.metrics            import (
                                    roc_auc_score,              # CLASSIFICATION ACCURACY METRIC
                                    mean_squared_error         # REGRESSION ACCURACY METRIC
                                )
from   sklearn.tree               import (
                                    DecisionTreeClassifier,     # CLASSIFICATION PREDICTIONS 
                                    DecisionTreeRegressor,      # REGRESSION PREDICTIONS
                                    export_text                # 
                                )
from   sklearn.ensemble           import RandomForestRegressor # ENEMBLE LEANING

###########################################################################################
## GET DATA
###########################################################################################
data_path  = "C:/Users/duffl/anaconda3/_CT161_GitHub_Repos/ml-zoomcamp-homework/07_midterm_project_plants/data/Australian_plant_translocation_database_PUBLIC_2018(ALL_TRANSLOCATIONS).csv"
df         = pd.read_csv(data_path)


# 2.0 - HELPER FUNCTIONS

In [2]:
# CLEAN STRINGS 
def fn_clean_text(value):
    if not isinstance(value, str):
        return value

    value = value.lower()
    value = value.replace('%', '_pct_')
    value = re.sub(r'[\s()/\-]+', '_', value)  # spaces, slashes, parentheses, hyphens
    value = re.sub(r"""['"]""", '', value)     # remove quotes
    value = re.sub(r'[,;.?]', '', value)       # remove punctuation
    return value
    
# CREATE A TABLE SHOWING HOW MANY ROWS ARE MISSING
def fn_col_value_status_table(data):
    if isinstance(data, pd.Series):
        data = data.to_frame()
    elif not isinstance(data, pd.DataFrame):
        raise TypeError("Pass a pandas Series or DataFrame")
        
    print("=" * 80)
    print("VALUES\t\tMISSING\t\t%_MISSING\tCOLUMN")
    print("=" * 80)

    for column_name in data.columns:
        non_missing = data[column_name].count()
        missing = data[column_name].isna().sum()
        total = non_missing + missing
        percent_missing = (missing / total * 100) if total else 0

        print(
            f"{non_missing}\t\t{missing}\t\t"
            f"{percent_missing:.2f} %\t\t{column_name}"
        )
    print("=" * 80)

# CREATE A TABLE SHOWING A COUNT OF HOW FREQUENTLY THE VALUES IN A FEATURE APPEAR
def fn_column_value_counts(data):
    if isinstance(data, pd.Series):
        columns = [(data.name or "value", data)]
    elif isinstance(data, pd.DataFrame):
        columns = [(name, data[name]) for name in data.columns]
    else:
        raise TypeError("Pass a pandas Series or DataFrame")

    for column_name, series in columns:
        counts = series.value_counts(dropna=False)
        total = len(series)

        print(f"\n{column_name}")
        print("COUNT\t\tFREQUENCY\tVALUE")
        print("=" * 60)

        for value, count in counts.items():
            percent = (count / total * 100) if total else 0
            print(f"{count}\t\t{percent:.2f}%\t\t{value}")

# REVIEW FEATURE DATA QUALITY
def fn_feature_data_quality(feature_name):
    fn_col_value_status_table(df[feature_name])
    fn_column_value_counts(df[feature_name])

# 3.0 - BEGIN PREPARING DATA

In [3]:
###########################################################################################
#   3.1 - CLEAN COLUMN NAMES AND RECORD VALUES
###########################################################################################
df.columns             = [fn_clean_text(column) for column in df.columns]
df                     = df.map(fn_clean_text)

# PREVIEW NEW  COLUMN NAMES 
fn_col_value_status_table(df)

VALUES		MISSING		%_MISSING	COLUMN
1182		0		0.00 %		status_summary
1182		0		0.00 %		translocation_number
1182		0		0.00 %		treatment_number
321		861		72.84 %		treatment_description
1182		0		0.00 %		species
1182		0		0.00 %		family
1182		0		0.00 %		lifeform
1182		0		0.00 %		state
1166		16		1.35 %		conservation_status_epbc_
1156		26		2.20 %		conservation_status_state_
1182		0		0.00 %		latitude_decimal_degrees_
1182		0		0.00 %		longitude_decimal_degrees_
1182		0		0.00 %		bioregion
1182		0		0.00 %		broad_habitat_group
1166		16		1.35 %		habitat_description
871		311		26.31 %		research_to_support_translocation_
1177		5		0.42 %		mitigation_translocation
1110		72		6.09 %		habitat_condition_eg_highly_disturbed_small_remnant_large_remnant_
1182		0		0.00 %		translocation_type
775		407		34.43 %		start_month
1138		44		3.72 %		start_year
892		290		24.53 %		number_of_subsequent_plantings
91		1091		92.30 %		number_of_seeds_direct_seeding_only_
1054		128		10.83 %		number_of_individuals_planted_or_germinate

In [4]:
# PREVIEW DATA VALUES 
df.head(20)

,status_summary,translocation_number,treatment_number,treatment_description,species,family,lifeform,state,conservation_status_epbc_,conservation_status_state_,...,short_term_success__pct__of_plants_which_survived_first_year_,medium_term_success_population_persisting_at_secure_site_,long_term_success_self_sustaining_population_established_with_dynamics_comparable_to_natural_populations_,reasons_for_failure_or_sub_optimal_performance_or_success_if_known_,failure_factors_summary,success_factors_summary,notes,project_managed_by,contact_person_s,references
0,incomplete_but_sufficient_for_analysis,10,1.0,NaN,abutilon_julianae,malvaceae,shrub,norfolk_island,cr,NaN,...,good,yes,yes,NaN,NaN,NaN,successful_part_of_restoration_planting_and_sp...,NaN,NaN,de_lange_pj_&_bgmurray_2003__chromosome_number...
1,incomplete_but_sufficient_for_analysis,11,1.0,NaN,abutilon_julianae,malvaceae,shrub,norfolk_island,cr,NaN,...,good,yes,yes,NaN,NaN,NaN,successful_part_of_restoration_planting_and_sp...,NaN,NaN,de_lange_pj_&_bgmurray_2003__chromosome_number...
2,finalised,854,1.0,propagule_type_seedlings_,acacia_aprica,fabaceae,shrub,wa,en,cr,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dbca_science_and_dbca_dbca_district,leonie_monks,monks_l_and_coates_d_2002_the_translocation_of...
3,finalised,854,2.0,propagule_type_seeds_,acacia_aprica,fabaceae,shrub,wa,en,cr,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dbca_science_and_dbca_dbca_district,leonie_monks,monks_l_and_coates_d_2002_the_translocation_of...
4,finalised,855,1.0,NaN,acacia_aprica,fabaceae,shrub,wa,en,cr,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dbca_science_and_dbca_district,leonie_monks_rebecca_dillon,monks_l_and_coates_d_2002_the_translocation_of...
5,finalised,262,1.0,NaN,acacia_attenuata,fabaceae,shrub,qld,vu,vu,...,10000,yes,yes,implementation_of_fire_management_plan_will_en...,NaN,planting_technique_active_disturbance_manageme...,brownlie_et_al_2009_paper:_the_maintenance_of_...,NaN,heather_brownlie_peter_duforq_alison_shapcott,usc_inquiry_into_environmental_offsets_brownli...
6,incomplete_but_sufficient_for_analysis,263,1.0,NaN,acacia_attenuata,fabaceae,shrub,qld,vu,vu,...,low,no_,no,planted_out_tube_stock_died_for_various_reason...,climate_dry__site_selection_lack_of_maintenance_,NaN,the_selected_site_was_not_ideal_but_was_the_on...,fraser_coast_regional_council,tina_raveneau,NaN
7,finalised,264,1.0,NaN,acacia_attenuata,fabaceae,shrub,qld,vu,vu,...,000,no,no,flooding_,climate_flooding__too_few_plants_to_bein_with,NaN,reason_for_translocation:_these_works_were_und...,department_of_transport_and_main_roads_opus_ve...,caroline_haskard_,haskard_c_2014_translocation_compliance_report...
8,finalised,265,1.0,NaN,acacia_attenuata,fabaceae,shrub,qld,vu,vu,...,000,no,no,inappropriate_translocation_habitat,poor_site_selection_lack_of_maintenance,NaN,sippy_creek_revegetation_site_ex_situ_populati...,NaN,robert_lamont,brownlie_h_playford_j_wallace_h_shapcott_a_200...
9,finalised,856,1.0,NaN,acacia_awestoniana,fabaceae,shrub,wa,vu,cr,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,dbca_science_and_dbca_district,leonie_monks_rebecca_dillon,NaN


In [5]:
###########################################################################################
#   3.2 - DROP RECORDS DATASET CREATORS HAVE MARKED AS UNSUITABLE FOR ANALYSIS
###########################################################################################
exclude_status_summary = r'incomplete|unable_to_monitor|finalised '

df                     = df.loc[
                          ~df['status_summary'].str.contains(
                            exclude_status_summary, 
                            na=False
                          )
                        ].copy()

In [6]:
###########################################################################################
#   3.3 - DROP FEATURES WHOSE CONTENT IS UNIQUE ACROSS ALL RECORDS OR OF LOW FREQUENCY 
###########################################################################################
exclude_features_list  =    [
                                'status_summary',
                                'treatment_description', 
                                'species', 
                                'family',
                                'bioregion',
                                'conservation_status_state_',
                                'latitude_decimal_degrees_',
                                'longitude_decimal_degrees_',
                                'habitat_description',
                                'start_month',
                                'start_year',
                                'number_of_seeds_direct_seeding_only_',
                                'number_of_individuals_planted_or_germinated_if_direct_seeding_total_',
                                'number_planted_notes',
                                'watering_in_only', 
                                'watering_<6_months_', 
                                'watering_>6_months_or_in_dry_season_', 
                                'last_monitoring_date',
                                'number_of_plants_surviving',
                                'number_of_subsequent_plantings',
                                'no_second_generation_plants',
                                'short_term_success__pct__of_plants_which_survived_first_year_', 
                                'medium_term_success_population_persisting_at_secure_site_',
                                'long_term_success_self_sustaining_population_established_with_dynamics_comparable_to_natural_populations_',
                                'reasons_for_failure_or_sub_optimal_performance_or_success_if_known_',
                                'failure_factors_summary',
                                'success_factors_summary',
                                'notes',
                                'project_managed_by',
                                'contact_person_s', 
                                'references' 
                            ]

missing                = [col for col in exclude_features_list if col not in df.columns]
print("Features to remove that were not found:\r\n", missing)
print()

# DROP COLUMNS IN LIST
df                     = df.drop(columns=exclude_features_list, errors='ignore')

# DEBUG SHOW NEW COLUMN NAMES
print("===================================================================================================================")
print("===== REMAINING FEATURES ==========================================================================================")
fn_col_value_status_table(df)


Features to remove that were not found:
 []

===== REMAINING FEATURES ==========================================================================================
VALUES		MISSING		%_MISSING	COLUMN
841		0		0.00 %		translocation_number
841		0		0.00 %		treatment_number
841		0		0.00 %		lifeform
841		0		0.00 %		state
830		11		1.31 %		conservation_status_epbc_
841		0		0.00 %		broad_habitat_group
665		176		20.93 %		research_to_support_translocation_
841		0		0.00 %		mitigation_translocation
838		3		0.36 %		habitat_condition_eg_highly_disturbed_small_remnant_large_remnant_
841		0		0.00 %		translocation_type
806		35		4.16 %		propagule_type_seed_seedling_cutting_tissue_culture_whole_plants_etc_
692		149		17.72 %		soil_preparation_y_n_
692		149		17.72 %		pre_planting_burn_y_n_
681		160		19.02 %		pre_planting_slashing_or_weeding_y_n_
690		151		17.95 %		pre_site_preparation_all_
684		157		18.67 %		herbivore_exclosure_fence_y_n_
684		157		18.67 %		herbivore_exclosure_cages_y_n_
684		157		18.67 %		tree_

In [7]:
# df.plants_surviving__pct__.head(40)

fn_col_value_status_table(df['broad_habitat_group'])

fn_column_value_counts(df['broad_habitat_group'])

VALUES		MISSING		%_MISSING	COLUMN
841		0		0.00 %		broad_habitat_group

broad_habitat_group
COUNT		FREQUENCY	VALUE
164		19.50%		rainforest
135		16.05%		southern_shrublands_heathlands_woodlands
95		11.30%		southern_shrublands_heathlands_woodlands_
82		9.75%		grassland
47		5.59%		mallee
47		5.59%		grassy_woodland
42		4.99%		grassland_with_chenopods
38		4.52%		wetland
36		4.28%		coastal_heath
21		2.50%		dry_sclerophyll
21		2.50%		wet_sclerophyll
20		2.38%		grassland_sub_tropical_
18		2.14%		montane
16		1.90%		coastal_shrubland
11		1.31%		semi_arid_woodland
9		1.07%		other
8		0.95%		banded_ironstone
7		0.83%		coastal_forest
6		0.71%		wetland_floodplain_
6		0.71%		dry_rainforest
5		0.59%		coastal_dunes
2		0.24%		grassland_lomandra_
2		0.24%		wetland_riparian_
2		0.24%		grassland_
1		0.12%		coastal_headland


In [8]:
# lifeform
fn_feature_data_quality('lifeform')


VALUES		MISSING		%_MISSING	COLUMN
841		0		0.00 %		lifeform

lifeform
COUNT		FREQUENCY	VALUE
394		46.85%		shrub
141		16.77%		tree
95		11.30%		perennial_forb
69		8.20%		ground_orchid
35		4.16%		perennial_vine
26		3.09%		annual_forb
22		2.62%		annual_grass
20		2.38%		perennial_grass
8		0.95%		cycad
7		0.83%		perennial_lily
5		0.59%		perennial_forb_fern_
5		0.59%		perennnial_forb
2		0.24%		orchid_epiphytic_
2		0.24%		grass_tree
2		0.24%		tree_mallee_
2		0.24%		fern
2		0.24%		epiphyte
1		0.12%		annual_sedge
1		0.12%		palm
1		0.12%		perennial_aquatic_forb
1		0.12%		shrub_


In [9]:
lifeform_map  =  {
                    'shrub': 'shrub',
                    'tree': 'tree',
                    'perennial_forb': 'perennial_forb',
                    'ground_orchid': 'ground_orchid',
                    'perennial_vine': 'perennial_vine',
                    'annual_forb': 'annual_forb',
                    'perennial_grass': 'perennial_grass',
                    'perennial_lily': 'perennial_lily',
                    'annual_grass': 'annual_grass',
                    'cycad': 'cycad',
                    'perennial_forb_fern_': 'perennial_forb_fern',
                    'perennial_sedge': 'perennial_sedge',
                    'perennnial_forb': 'perennnial_forb',
                    'perennial_sedge': 'perennial_sedge',
                    'perennnial_forb': 'perennnial_forb',
                    'orchid_epiphytic_': 'orchid_epiphytic',
                    'palm': 'palm',
                    'grass_tree': 'grass_tree',
                    'tree_mallee_': 'tree_mallee',
                    'fern': 'fern',
                    'epiphyte': 'epiphyte',
                    'annual_sedge': 'annual_sedge',
                    'tree_': 'tree',
                    'perennial_aquatic_forb': 'perennial_aquatic_forb',
                    'shrub_' : 'shrub'
                  }
df['col_lifeform']     = df['lifeform'].astype('string').replace(lifeform_map).fillna('unk')
df.drop(columns=['lifeform'], inplace=True)

In [10]:
# state
fn_feature_data_quality('state')

VALUES		MISSING		%_MISSING	COLUMN
841		0		0.00 %		state

state
COUNT		FREQUENCY	VALUE
207		24.61%		nsw
196		23.31%		vic
186		22.12%		sa
133		15.81%		wa
91		10.82%		qld
20		2.38%		tas
6		0.71%		act
2		0.24%		qld_


In [11]:
df['state'] = df['state'].replace('qld_', 'qld')
# fn_feature_data_quality('state')


In [12]:
# conservation_status_epbc_
fn_feature_data_quality('conservation_status_epbc_')

VALUES		MISSING		%_MISSING	COLUMN
830		11		1.31 %		conservation_status_epbc_

conservation_status_epbc_
COUNT		FREQUENCY	VALUE
415		49.35%		en
175		20.81%		not_listed
150		17.84%		vu
66		7.85%		cr
11		1.31%		nan
10		1.19%		vu_
6		0.71%		rotap_2rc_
4		0.48%		not_listed_rotap_3rca_
3		0.36%		not_listed_rotap_
1		0.12%		ev


In [13]:
feature_name = 'conservation_status_epbc_' 

df[feature_name] = df[feature_name].replace('vu_', 'vu')
df[feature_name] = df[feature_name].replace('rotap_2rc_', 'rotap_2rc')
df[feature_name] = df[feature_name].replace('not_listed_rotap_3rca_', 'not_listed_rotap_3rca')
df[feature_name] = df[feature_name].replace('not_listed_rotap_', 'not_listed_rotap_')
    
#fn_feature_data_quality(feature_name)


In [14]:
source_column          = 'research_to_support_translocation_'
destination_column     = 'col_research'

is_empty               = df[column].isna() | df[column].astype('string').str.strip().eq('')
is_underscore          = df[column].astype('string').str.strip().eq('_')

df[destination_column] = 'yes'
df.loc[is_empty | is_underscore, destination_column] = 'no'

df.drop(columns=['source_column'], inplace=True)


NameError: name 'column' is not defined

In [ ]:
# research_to_support_translocation_
fn_feature_data_quality('research_to_support_translocation_')

In [ ]:
# mitigation_translocation
fn_feature_data_quality('mitigation_translocation')

In [ ]:
feature_name = 'mitigation_translocation' 

df[feature_name] = df[feature_name].replace('y_seedlings_damaged_by_maintenance_work_in_the_area', 'y')
# fn_feature_data_quality(feature_name)

In [ ]:
fn_col_value_status_table(df)

In [ ]:
mitigation_map = {
                    'Y - seedlings damaged by maintenance work in the area' : 'yes',
                    'y' : 'yes',
                    'n' : 'no',
                    '?' : 'no'
                }
# df.mitigation_translocation = df.mitigation_translocation.astype('string').map(mitigation_map).fillna('unk')
df.mitigation_translocation = df.mitigation_translocation.astype('string').replace(mitigation_map).fillna('unk')

In [ ]:
# mitigation_translocation
fn_feature_data_quality('broad_habitat_group')

In [ ]:
habitat_map  =  {
                    'rainforest': 'rainforest',
                    'southern_shrublands_heathlands_woodlands': 'southern_shrublands_heathlands_woodlands',
                    'southern_shrublands_heathlands_woodlands_': 'southern_shrublands_heathlands_woodlands',
                    'grassland': 'grassland',
                    'grassy_woodland': 'grassy_woodland',
                    'wetland': 'wetland',
                    'dry_sclerophyll': 'dry_sclerophyll',
                    'mallee': 'mallee',
                    'grassland_with_chenopods': 'grassland_with_chenopods',
                    'coastal_heath': 'coastal_heath',
                    'wet_sclerophyll': 'wet_sclerophyll',
                    'banded_ironstone': 'banded_ironstone',
                    'grassland_sub_tropical_': 'grassland_sub_tropical',
                    'montane': 'montane',
                    'other': 'other',
                    'coastal_shrubland': 'coastal_shrubland',
                    'semi_arid_woodland': 'semi_arid_woodland',
                    'wetland_floodplain_': 'wetland_floodplain',
                    'coastal_headland': 'coastal_headland',
                    'coastal_forest': 'coastal_forest',
                    'coastal_dunes': 'coastal_dunes',
                    'grassland_': 'grassland',
                    'grassland_lomandra_': 'grassland_lomandra',
                    'wetland_riparian_': 'wetland_riparian',
                  }
df['broad_habitat_group']     = df['broad_habitat_group'].astype('string').replace(habitat_map).fillna('unk')
# df.drop(columns=['broad_habitat_group'], inplace=True)

fn_feature_data_quality('broad_habitat_group')

In [ ]:
fn_feature_data_quality('plants_surviving__pct__')

In [ ]:
##########################################################################################
# NORMALIZE PERCENTAGE OF PLANTS THAT SURVIVED TRANSPLANTING
###########################################################################################
source = df['plants_surviving__pct__'].astype('string').str.strip()

replacements = {
    '<NA>'                          : '0.00',
    '<na>'                          : '0.00',
    '??'                            : '0.00',
    '#div/0!'                       : '0.00',
    '#ref!'                         : '0.00',
    '#value!'                       : '0.00',
    'apr_00'                        : '0.00',
    'n_a_annual_'                   : '0.00',
    'n_a_for_annual_species'        : '0.00',
    'n_a_annual_plant_'             : '0.00',
    'n_a_direct_seeding_'           : '0.00',
    'n_a_short_lived_'              : '0.00',
    'Not monitored yet'             : '0.00',
    'unknown_probably_0_'           : '0.00',
    '0_but_substantial_recruitment_': '0.00',
    '<5'                            : '5.00',
    '<10'                           : '10.00',
    '>10'                           : '10.00',
    '>10_pct_'                      : '10.00',
    '16_pct_'                       : '16.00',
    '4_pct_'                        : '4.00', 
    '11_pct_'                       : '11.00',
    'c20'                           : '20.00',
    'c25_pct_'                      : '25.00',
    '17_pct_'                       : '17.00',
    '<20'                           : '20.00',
    'c.20'                          : '20.00',
    '>25'                           : '25.00',
    '>25_pct_'                      : '25.00',
    'c.25'                          : '25.00',
    '20-40_pct_'                    : '30.00',
    '20-40'                         : '30.00',
    'c.25'                          : '25.00',
    '29_pct_'                       : '29.00',
    '~30'                           : '30.00',
    '>30'                           : '30.00',
    '>33'                           : '33.00',
    '~40'                           : '40.00',
    '>40'                           : '40.00',
    '<50?'                          : '50.00',
    '<50_pct_'                      : '50.00',
    '<50_pct_?'                     : '50.00',
    '<50?'                          : '50.00',
    '50_pct_'                       : '50.00',
    '~50'                           : '50.00',
    '>50_pct_'                      : '50.00',
    '>50'                           : '50.00',
    '>50_pct_'                      : '50.00',
    '52_pct_'                       : '52.00',
    '60_pct_'                       : '60.00',
    '69_pct_'                       : '69.00',
    'unknown_but_probably_>75_pct_' : '75.00',
    '77_pct_'                       : '77.00',
    '80_pct_'                       : '80.0',
    '80_pct__across_all_sites_and_treatments_'  : '80.00',
    '80_pct_ (across all sites and treatments)' : '80.00',
    '83_pct_'                       : '83.00',
    '85_pct_'                       : '85.0',
    '>90'                           : '90.00',
    '>95'                           : '95.00',
    '>95_pct_'                      : '95.00',
    '100_pct_'                      : '100.00',
    '>100_with_recruitment_'        : '0.00',
    'good'                          : '0.00',
    'unknown'                       : '0.00',
    'Unknown'                       : '0.00',
    'unknown_number_germinating_not_documented_' : '0.00',
    'unknown_how_many_were_planted_in_total'     : '0.00',
    'unknown_number_of_germinants_not_counted_'  : '0.00',
    'too_early'                     : '0.00',
    'plants_in_the_ground_for_<_12_months'       : '0.00',
    '<12_months_since_planting'     : '0.00',
    '<_12_mnths_since_translocation': '0.00',
    '<12_months_elapsed'            : '0.00',
    '_pct_'                         : '0.00',
}

cleaned = (
    source.replace(replacements)
          .str.replace('%', '', regex=False)
          .str.strip()
)

parts = cleaned.str.split(':',n=1)
print(df['plants_surviving__pct__'], parts)

# CREATE NEW COLUMN
df['col_plants_surviving_cnt'] = (
    pd.to_numeric(parts.str[0], errors='coerce')
      .fillna(0.0)
)

df['col_plants_surviving_pct'] = (
    pd.to_numeric(parts.str[1], errors='coerce')
      .fillna(0.0)
)
# df 'col_plants_surviving_pct'] = parts.str[0].astype('string').str.strip()
# df['col_plants_surviving_ct'] = parts.str[1].astype('string').str.strip()

print("===== 'col_plants_surviving_pct' cleaned ===============")


# print(f"===== {cleaned[values.isna()].value_counts(dropna=False)} ===============")


print(f"\r\n\r\ndf['col_plants_surviving_cnt'].unique\t\t{df['col_plants_surviving_cnt'].unique}")
print(f"\r\n\r\ndf['col_plants_surviving_pct'].unique\t\t{df['col_plants_surviving_pct'].unique}")

In [ ]:
# DEBUG 
col_list = [
    'col_plants_surviving_pct'
]
for col in col_list:
    print(f"{col} ({df[col].nunique()} unique values):")
    for value, count in df[col].value_counts(dropna=False).items():
        print(f"    {value}: {count}")

In [ ]:
df[['plants_surviving__pct__','col_plants_surviving_cnt','col_plants_surviving_pct']].head(30)

In [ ]:
df.col_plants_surviving_pct.head()